In [1]:
from FCN_model import FCN, FCN_tune
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True

In [14]:
origin_FCN = FCN(in_channels = 1, num_classes=2).to("cuda")

In [15]:
origin_FCN

FCN(
  (layers): Sequential(
    (0): Conv1d(1, 128, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=same)
    (4): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=same)
    (7): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (8): ReLU()
    (9): AdaptiveAvgPool1d(output_size=1)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [16]:
epochs = 2000
optimizer = torch.optim.Adam(origin_FCN.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=50,
    min_lr=0.0001
)

#### Test with original dataset without z-score again

In [2]:
FordA_train = np.load("FordA_processed/FordA_TRAIN.npy")
FordA_test = np.load("FordA_processed/FordA_TEST.npy")

In [18]:
FordA_X_train = np.expand_dims(FordA_train[:, 1:], axis=1)
FordA_y_train = FordA_train[:, 0]
FordA_y_train = (FordA_y_train == 1).astype(np.int64)

In [19]:
print(FordA_X_train.shape)
print(FordA_y_train.shape)

(3601, 1, 500)
(3601,)


In [20]:
FordA_X_test = np.expand_dims(FordA_test[:, 1:], axis=1)
FordA_y_test = FordA_test[:, 0]
FordA_y_test = (FordA_y_test == 1).astype(np.int64)

In [10]:
FordA_X_train = torch.tensor(FordA_X_train, dtype=torch.float32)
FordA_y_train = torch.tensor(FordA_y_train, dtype=torch.long)

In [11]:
FordA_X_test = torch.tensor(FordA_X_test, dtype=torch.float32)
FordA_y_test = torch.tensor(FordA_y_test, dtype=torch.long)

In [12]:
dataset = TensorDataset(FordA_X_train, FordA_y_train)

train_loader = DataLoader(
    dataset,
    batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
    shuffle=True
)

#### Training experiment 1

In [13]:
metrics = {
    "epoch": [],
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

In [14]:
min_loss = 0.2
for epoch in range(epochs):
    metrics["epoch"].append(epoch)
    origin_FCN.train()
    running_loss = 0.0
    total_samples = 0
    all_predictions = []
    all_labels = []

    for X, y in train_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        optimizer.zero_grad()
        # print(batch_features.shape)
        # Forward pass
        outputs = origin_FCN(X)
        loss = criterion(outputs, y)

        # print(outputs)
        # print(loss)

         # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        predictions = outputs.argmax(dim=1)
        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )
        all_labels.extend(
            y.cpu().numpy()
        )
    epoch_loss = running_loss / total_samples

    accuracy = accuracy_score(all_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
                                                        all_labels,
                                                        all_predictions,
                                                        average="macro",
                                                        zero_division=0
                                                    )
    # Save metrics
    if epoch_loss <= min_loss:
        torch.save({
                "model_state_dict": origin_FCN.state_dict(),
                "metrics": metrics,
                "optimizer_state_dict": optimizer.state_dict()
            }, f"trained_model/fcn_model_min_loss_epoch{epoch}_loss{epoch_loss: .5f}.pth")
        # torch.save(origin_FCN.state_dict(), f"fcn_model_highest_accuracy_epoch{epoch}.pth")
        min_loss = epoch_loss
    metrics["loss"].append(epoch_loss)
    metrics["accuracy"].append(accuracy)
    metrics["precision"].append(precision)
    metrics["recall"].append(recall)
    metrics["f1"].append(f1)

    print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Accuracy: {accuracy:.4f} | "
            f"Precision: {precision:.4f} | "
            f"Recall: {recall:.4f} | "
            f"F1: {f1:.4f}"
    )
    #     break
    # break
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv("FCN_original_metrics.csv")

c:\Users\96561\miniconda3\envs\UCR-pytorch\Lib\site-packages\torch\nn\modules\conv.py:380: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1105.)
  return F.conv1d(


Epoch 1/2000 | Loss: 0.5498 | Accuracy: 0.7148 | Precision: 0.7560 | Recall: 0.7197 | F1: 0.7057
Epoch 2/2000 | Loss: 0.4445 | Accuracy: 0.7998 | Precision: 0.7999 | Recall: 0.7993 | F1: 0.7995
Epoch 3/2000 | Loss: 0.3960 | Accuracy: 0.8364 | Precision: 0.8364 | Recall: 0.8365 | F1: 0.8364
Epoch 4/2000 | Loss: 0.3818 | Accuracy: 0.8387 | Precision: 0.8398 | Recall: 0.8394 | F1: 0.8386
Epoch 5/2000 | Loss: 0.3715 | Accuracy: 0.8406 | Precision: 0.8412 | Recall: 0.8411 | F1: 0.8406
Epoch 6/2000 | Loss: 0.3460 | Accuracy: 0.8648 | Precision: 0.8657 | Recall: 0.8641 | F1: 0.8645
Epoch 7/2000 | Loss: 0.3425 | Accuracy: 0.8748 | Precision: 0.8762 | Recall: 0.8740 | F1: 0.8744
Epoch 8/2000 | Loss: 0.3175 | Accuracy: 0.8814 | Precision: 0.8821 | Recall: 0.8809 | F1: 0.8812
Epoch 9/2000 | Loss: 0.3140 | Accuracy: 0.8875 | Precision: 0.8884 | Recall: 0.8869 | F1: 0.8873
Epoch 10/2000 | Loss: 0.3089 | Accuracy: 0.8781 | Precision: 0.8781 | Recall: 0.8783 | F1: 0.8781
Epoch 11/2000 | Loss: 0.2940 

In [16]:
metrics_df.sort_values(by="loss")

,epoch,loss,accuracy,precision,recall,f1
1905,1905,0.111185,0.963621,0.963556,0.963661,0.963602
1887,1887,0.113149,0.963066,0.963208,0.962909,0.963031
1891,1891,0.113252,0.962510,0.962520,0.962451,0.962484
1890,1890,0.113446,0.961400,0.961340,0.961424,0.961379
1997,1997,0.113955,0.960844,0.960868,0.960770,0.960815
...,...,...,...,...,...,...
4,4,0.371503,0.840600,0.841154,0.841144,0.840600
3,3,0.381829,0.838656,0.839829,0.839417,0.838643
2,2,0.395996,0.836434,0.836354,0.836547,0.836389
1,1,0.444533,0.799778,0.799920,0.799320,0.799483


#### Inference on test

In [17]:
origin_FCN = FCN(in_channels = 1, num_classes=2).to("cuda")

In [18]:
test_dataset = TensorDataset(FordA_X_test, FordA_y_test)

test_loader = DataLoader(
    test_dataset,
    batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
    shuffle=False
)

In [19]:
state_dict = torch.load(
    "fcn_model_highest_accuracy_epoch1905.pth",
    map_location="cuda",
    weights_only=True
)

In [20]:
origin_FCN.load_state_dict(state_dict["model_state_dict"])
origin_FCN = origin_FCN.to("cuda")
origin_FCN.eval()

FCN(
  (layers): Sequential(
    (0): Conv1d(1, 128, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=same)
    (4): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=same)
    (7): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (8): ReLU()
    (9): AdaptiveAvgPool1d(output_size=1)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [21]:
running_loss = 0.0
total_samples = 0
all_predictions = []
all_labels = []

test_metrics = {
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for X, y in test_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        # Forward pass
        outputs = origin_FCN(X)
        loss = criterion(outputs, y)

        # Predictions
        predictions = torch.argmax(outputs, dim=1)

        # Accumulate loss
        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        # Collect predictions and labels
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
# Compute evaluation metrics
test_loss = running_loss / total_samples
accuracy = accuracy_score(all_labels, all_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

# Store metrics
test_metrics = {
    "loss": test_loss,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "predictions": all_predictions,
    "labels": all_labels
}

print(
    f"Test | "
    f"Loss: {test_loss:.4f} | "
    f"Accuracy: {accuracy:.4f} | "
    f"Precision: {precision:.4f} | "
    f"Recall: {recall:.4f} | "
    f"F1: {f1:.4f}"
)

Test | Loss: 0.1992 | Accuracy: 0.9273 | Precision: 0.9276 | Recall: 0.9279 | F1: 0.9273


#### Experiment 2 with another z-score

In [2]:
origin_FCN = FCN(in_channels = 1, num_classes=2).to("cuda")

In [3]:
epochs = 2000
optimizer = torch.optim.Adam(origin_FCN.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=50,
    min_lr=0.0001
)

In [17]:
FordA_train = np.load("FordA_processed/FordA_TRAIN.npy")
FordA_test = np.load("FordA_processed/FordA_TEST.npy")

In [18]:
FordA_X_train = np.expand_dims(FordA_train[:, 1:], axis=1)
FordA_y_train = FordA_train[:, 0]
FordA_y_train = (FordA_y_train == 1).astype(np.int64)

In [19]:
FordA_X_test = np.expand_dims(FordA_test[:, 1:], axis=1)
FordA_y_test = FordA_test[:, 0]
FordA_y_test = (FordA_y_test == 1).astype(np.int64)

In [20]:
x_train_mean = FordA_X_train.mean()
x_train_std = FordA_X_train.std()
FordA_X_train = (FordA_X_train - x_train_mean)/(x_train_std)
FordA_X_test = (FordA_X_test - x_train_mean)/(x_train_std)

In [8]:
FordA_X_train = torch.tensor(FordA_X_train, dtype=torch.float32)
FordA_y_train = torch.tensor(FordA_y_train, dtype=torch.long)

In [21]:
FordA_X_test = torch.tensor(FordA_X_test, dtype=torch.float32)
FordA_y_test = torch.tensor(FordA_y_test, dtype=torch.long)

In [10]:
dataset = TensorDataset(FordA_X_train, FordA_y_train)

train_loader = DataLoader(
    dataset,
    batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
    shuffle=True
)

In [11]:
metrics = {
    "epoch": [],
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

In [12]:
min_loss = 0.2
for epoch in range(epochs):
    metrics["epoch"].append(epoch)
    origin_FCN.train()
    running_loss = 0.0
    total_samples = 0
    all_predictions = []
    all_labels = []

    for X, y in train_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        optimizer.zero_grad()
        # print(batch_features.shape)
        # Forward pass
        outputs = origin_FCN(X)
        loss = criterion(outputs, y)

        # print(outputs)
        # print(loss)

         # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        predictions = outputs.argmax(dim=1)
        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )
        all_labels.extend(
            y.cpu().numpy()
        )
    epoch_loss = running_loss / total_samples

    accuracy = accuracy_score(all_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
                                                        all_labels,
                                                        all_predictions,
                                                        average="macro",
                                                        zero_division=0
                                                    )
    # Save metrics
    if epoch_loss <= min_loss:
        torch.save({
                "model_state_dict": origin_FCN.state_dict(),
                "metrics": metrics,
                "optimizer_state_dict": optimizer.state_dict()
            }, f"trained_model/fcn_model_zscored_min_loss_epoch{epoch}_loss{epoch_loss: .5f}.pth")
        # torch.save(origin_FCN.state_dict(), f"fcn_model_highest_accuracy_epoch{epoch}.pth")
        min_loss = epoch_loss
    metrics["loss"].append(epoch_loss)
    metrics["accuracy"].append(accuracy)
    metrics["precision"].append(precision)
    metrics["recall"].append(recall)
    metrics["f1"].append(f1)

    print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Accuracy: {accuracy:.4f} | "
            f"Precision: {precision:.4f} | "
            f"Recall: {recall:.4f} | "
            f"F1: {f1:.4f}"
    )
    #     break
    # break
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv("FCN_original_metrics_zscored.csv")

c:\Users\96561\miniconda3\envs\UCR-pytorch\Lib\site-packages\torch\nn\modules\conv.py:380: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1105.)
  return F.conv1d(


Epoch 1/2000 | Loss: 0.5498 | Accuracy: 0.7140 | Precision: 0.7555 | Recall: 0.7189 | F1: 0.7047
Epoch 2/2000 | Loss: 0.4446 | Accuracy: 0.8017 | Precision: 0.8018 | Recall: 0.8013 | F1: 0.8014
Epoch 3/2000 | Loss: 0.3955 | Accuracy: 0.8381 | Precision: 0.8380 | Recall: 0.8382 | F1: 0.8381
Epoch 4/2000 | Loss: 0.3817 | Accuracy: 0.8362 | Precision: 0.8375 | Recall: 0.8370 | F1: 0.8361
Epoch 5/2000 | Loss: 0.3711 | Accuracy: 0.8395 | Precision: 0.8401 | Recall: 0.8401 | F1: 0.8395
Epoch 6/2000 | Loss: 0.3476 | Accuracy: 0.8634 | Precision: 0.8641 | Recall: 0.8628 | F1: 0.8631
Epoch 7/2000 | Loss: 0.3433 | Accuracy: 0.8731 | Precision: 0.8747 | Recall: 0.8723 | F1: 0.8727
Epoch 8/2000 | Loss: 0.3165 | Accuracy: 0.8820 | Precision: 0.8825 | Recall: 0.8815 | F1: 0.8818
Epoch 9/2000 | Loss: 0.3108 | Accuracy: 0.8898 | Precision: 0.8907 | Recall: 0.8892 | F1: 0.8895
Epoch 10/2000 | Loss: 0.3039 | Accuracy: 0.8850 | Precision: 0.8850 | Recall: 0.8852 | F1: 0.8850
Epoch 11/2000 | Loss: 0.2870 

In [13]:
metrics_df.sort_values(by="loss")

,epoch,loss,accuracy,precision,recall,f1
1898,1898,0.110408,0.962233,0.962209,0.962209,0.962209
1905,1905,0.110873,0.963621,0.963544,0.963689,0.963604
1897,1897,0.112210,0.960567,0.960487,0.960738,0.960554
1891,1891,0.113042,0.961122,0.961086,0.961111,0.961098
1899,1899,0.113830,0.959178,0.959293,0.959033,0.959141
...,...,...,...,...,...,...
4,4,0.371081,0.839489,0.840111,0.840061,0.839489
3,3,0.381653,0.836157,0.837493,0.836965,0.836137
2,2,0.395519,0.838101,0.838013,0.838201,0.838053
1,1,0.444600,0.801722,0.801839,0.801286,0.801443


#### inference on test

In [14]:
origin_FCN = FCN(in_channels = 1, num_classes=2).to("cuda")

In [22]:
test_dataset = TensorDataset(FordA_X_test, FordA_y_test)

test_loader = DataLoader(
    test_dataset,
    batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
    shuffle=False
)

In [23]:
state_dict = torch.load(
    "fcn_model_zscored_min_loss_epoch1898_loss 0.11041.pth",
    map_location="cuda",
    weights_only=True
)

In [24]:
origin_FCN.load_state_dict(state_dict["model_state_dict"])
origin_FCN = origin_FCN.to("cuda")
origin_FCN.eval()

FCN(
  (layers): Sequential(
    (0): Conv1d(1, 128, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=same)
    (4): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=same)
    (7): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (8): ReLU()
    (9): AdaptiveAvgPool1d(output_size=1)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [25]:
running_loss = 0.0
total_samples = 0
all_predictions = []
all_labels = []

test_metrics = {
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for X, y in test_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        # Forward pass
        outputs = origin_FCN(X)
        loss = criterion(outputs, y)

        # Predictions
        predictions = torch.argmax(outputs, dim=1)

        # Accumulate loss
        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        # Collect predictions and labels
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
# Compute evaluation metrics
test_loss = running_loss / total_samples
accuracy = accuracy_score(all_labels, all_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

# Store metrics
test_metrics = {
    "loss": test_loss,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "predictions": all_predictions,
    "labels": all_labels
}

print(
    f"Test | "
    f"Loss: {test_loss:.4f} | "
    f"Accuracy: {accuracy:.4f} | "
    f"Precision: {precision:.4f} | "
    f"Recall: {recall:.4f} | "
    f"F1: {f1:.4f}"
)

Test | Loss: 0.2254 | Accuracy: 0.9205 | Precision: 0.9216 | Recall: 0.9215 | F1: 0.9205


#### optuna tuned training no zscore

{'filters1': 256, 'filters2': 128, 'filters3': 64, 'kernel1': 8, 'kernel2': 10, 'kernel3': 6, 'learning_rate': 0.004038721483970501, 'batch_size': 64, 'weight_decay': 0.0, 'optimizer': 'AdamW'}

In [3]:
FordA_train = np.load("FordA_processed/FordA_TRAIN.npy")
FordA_test = np.load("FordA_processed/FordA_TEST.npy")

In [4]:
FordA_X_train = np.expand_dims(FordA_train[:, 1:], axis=1)
FordA_y_train = FordA_train[:, 0]
FordA_y_train = (FordA_y_train == 1).astype(np.int64)

FordA_X_test = np.expand_dims(FordA_test[:, 1:], axis=1)
FordA_y_test = FordA_test[:, 0]
FordA_y_test = (FordA_y_test == 1).astype(np.int64)


In [5]:
FordA_X_train = torch.tensor(FordA_X_train, dtype=torch.float32)
FordA_y_train = torch.tensor(FordA_y_train, dtype=torch.long)
FordA_X_test = torch.tensor(FordA_X_test, dtype=torch.float32)
FordA_y_test = torch.tensor(FordA_y_test, dtype=torch.long)

In [6]:
dataset = TensorDataset(FordA_X_train, FordA_y_train)

train_loader = DataLoader(
    dataset,
    batch_size=64,
)

In [7]:
tuned_FCN = FCN_tune(
    in_channels=1,
    num_classes=2,
    filters=[256, 128, 64],
    kernels=[8,10,6]
).to("cuda")

In [8]:
epochs = 2000
optimizer = torch.optim.AdamW(tuned_FCN.parameters(), lr= 0.004038721483970501, weight_decay=0)

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=50,
    min_lr=0.0001
)

In [9]:
metrics = {
    "epoch": [],
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

In [10]:
min_loss = 0.2
for epoch in range(epochs):
    metrics["epoch"].append(epoch)
    tuned_FCN.train()
    running_loss = 0.0
    total_samples = 0
    all_predictions = []
    all_labels = []

    for X, y in train_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        optimizer.zero_grad()
        # print(batch_features.shape)
        # Forward pass
        outputs = tuned_FCN(X)
        loss = criterion(outputs, y)

        # print(outputs)
        # print(loss)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        predictions = outputs.argmax(dim=1)
        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )
        all_labels.extend(
            y.cpu().numpy()
        )
    epoch_loss = running_loss / total_samples

    accuracy = accuracy_score(all_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
                                                        all_labels,
                                                        all_predictions,
                                                        average="macro",
                                                        zero_division=0
                                                    )
    # Save metrics
    if epoch_loss <= min_loss:
        torch.save({
                "model_state_dict": tuned_FCN.state_dict(),
                "metrics": metrics,
                "optimizer_state_dict": optimizer.state_dict()
            }, f"trained_model/fcn_tuned_model_min_loss_epoch{epoch}_loss{epoch_loss: .5f}.pth")
        # torch.save(origin_FCN.state_dict(), f"fcn_model_highest_accuracy_epoch{epoch}.pth")
        min_loss = epoch_loss

    metrics["loss"].append(epoch_loss)
    metrics["accuracy"].append(accuracy)
    metrics["precision"].append(precision)
    metrics["recall"].append(recall)
    metrics["f1"].append(f1)

    print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Accuracy: {accuracy:.4f} | "
            f"Precision: {precision:.4f} | "
            f"Recall: {recall:.4f} | "
            f"F1: {f1:.4f}"
    )
    #     break
    # break
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv("FCN_tuned_metrics.csv")

c:\Users\96561\miniconda3\envs\UCR-pytorch\Lib\site-packages\torch\nn\modules\conv.py:380: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1105.)
  return F.conv1d(


Epoch 1/2000 | Loss: 0.4073 | Accuracy: 0.7981 | Precision: 0.7993 | Recall: 0.7989 | F1: 0.7981
Epoch 2/2000 | Loss: 0.2610 | Accuracy: 0.8889 | Precision: 0.8892 | Recall: 0.8886 | F1: 0.8888
Epoch 3/2000 | Loss: 0.2470 | Accuracy: 0.8914 | Precision: 0.8915 | Recall: 0.8912 | F1: 0.8913
Epoch 4/2000 | Loss: 0.2333 | Accuracy: 0.9042 | Precision: 0.9045 | Recall: 0.9039 | F1: 0.9041
Epoch 5/2000 | Loss: 0.2176 | Accuracy: 0.9109 | Precision: 0.9112 | Recall: 0.9105 | F1: 0.9107
Epoch 6/2000 | Loss: 0.2121 | Accuracy: 0.9142 | Precision: 0.9143 | Recall: 0.9140 | F1: 0.9141
Epoch 7/2000 | Loss: 0.2048 | Accuracy: 0.9159 | Precision: 0.9159 | Recall: 0.9157 | F1: 0.9158
Epoch 8/2000 | Loss: 0.2006 | Accuracy: 0.9186 | Precision: 0.9188 | Recall: 0.9184 | F1: 0.9185
Epoch 9/2000 | Loss: 0.1994 | Accuracy: 0.9192 | Precision: 0.9195 | Recall: 0.9189 | F1: 0.9191
Epoch 10/2000 | Loss: 0.1959 | Accuracy: 0.9200 | Precision: 0.9203 | Recall: 0.9197 | F1: 0.9199
Epoch 11/2000 | Loss: 0.1928 

In [11]:
metrics_df.sort_values(by="loss")

,epoch,loss,accuracy,precision,recall,f1
1309,1309,1.224866e-09,1.000000,1.000000,1.000000,1.000000
1308,1308,1.257971e-09,1.000000,1.000000,1.000000,1.000000
1304,1304,1.423493e-09,1.000000,1.000000,1.000000,1.000000
1306,1306,1.423493e-09,1.000000,1.000000,1.000000,1.000000
1305,1305,1.423493e-09,1.000000,1.000000,1.000000,1.000000
...,...,...,...,...,...,...
824,824,3.629627e-01,0.948070,0.947992,0.948114,0.948044
825,825,3.965609e-01,0.917801,0.917707,0.917846,0.917763
0,0,4.072965e-01,0.798112,0.799253,0.798860,0.798094
1816,1816,5.426722e-01,0.913080,0.913090,0.912947,0.913010


In [12]:
test_dataset = TensorDataset(FordA_X_test, FordA_y_test)

test_loader = DataLoader(
    test_dataset,
    batch_size=64, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
    shuffle=False
)

In [ ]:
tuned_FCN = FCN_tune(
    in_channels=1,
    num_classes=2,
    filters=[256, 128, 64],
    kernels=[8,10,6]
).to("cuda")
state_dict = torch.load(
    "trained_model/fcn_tuned_model_min_loss_epoch407_loss 0.00409.pth",
    map_location="cuda",
    weights_only=True
)
tuned_FCN.load_state_dict(state_dict["model_state_dict"])
tuned_FCN = tuned_FCN.to("cuda")
tuned_FCN.eval()

In [17]:
running_loss = 0.0
total_samples = 0
all_predictions = []
all_labels = []

test_metrics = {
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for X, y in test_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        # Forward pass
        outputs = tuned_FCN(X)
        loss = criterion(outputs, y)

        # Predictions
        predictions = torch.argmax(outputs, dim=1)

        # Accumulate loss
        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        # Collect predictions and labels
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
# Compute evaluation metrics
test_loss = running_loss / total_samples
accuracy = accuracy_score(all_labels, all_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

# Store metrics
test_metrics = {
    "loss": test_loss,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "predictions": all_predictions,
    "labels": all_labels
}

print(
    f"Test | "
    f"Loss: {test_loss:.4f} | "
    f"Accuracy: {accuracy:.4f} | "
    f"Precision: {precision:.4f} | "
    f"Recall: {recall:.4f} | "
    f"F1: {f1:.4f}"
)

Test | Loss: 0.6632 | Accuracy: 0.8909 | Precision: 0.9015 | Recall: 0.8884 | F1: 0.8896


#### use a validation set to get early stopping

In [53]:
FordA_train = np.load("FordA_processed/FordA_TRAIN.npy")
FordA_test = np.load("FordA_processed/FordA_TEST.npy")
FordA_X_train = np.expand_dims(FordA_train[:, 1:], axis=1)
FordA_y_train = FordA_train[:, 0]
FordA_y_train = (FordA_y_train == 1).astype(np.int64)

FordA_X_test = np.expand_dims(FordA_test[:, 1:], axis=1)
FordA_y_test = FordA_test[:, 0]
FordA_y_test = (FordA_y_test == 1).astype(np.int64)


In [54]:
FordA_X_train = torch.tensor(FordA_X_train, dtype=torch.float32)
FordA_y_train = torch.tensor(FordA_y_train, dtype=torch.long)
FordA_X_test = torch.tensor(FordA_X_test, dtype=torch.float32)
FordA_y_test = torch.tensor(FordA_y_test, dtype=torch.long)

In [55]:
from sklearn.model_selection import train_test_split
train_idx, val_idx = train_test_split(
    np.arange(len(FordA_y_train)),
    test_size=0.1,
    stratify=FordA_y_train.numpy(),
    random_state=42
)

train_ds = TensorDataset(FordA_X_train[train_idx], FordA_y_train[train_idx])
val_ds = TensorDataset(FordA_X_train[val_idx], FordA_y_train[val_idx])
test_ds = TensorDataset(FordA_X_test, FordA_y_test)

In [65]:
# dataset = TensorDataset(FordA_X_train, FordA_y_train)

train_loader = DataLoader(
    train_ds,
    batch_size=64, 
    shuffle=False
)

val_loader = DataLoader(
    val_ds,
    batch_size=64, 
    shuffle=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=64, 
    shuffle=False
)

In [57]:
tuned_FCN = FCN_tune(
    in_channels=1,
    num_classes=2,
    filters=[256, 128, 64],
    kernels=[8,10,6]
).to("cuda")

In [58]:
epochs = 2000
optimizer = torch.optim.AdamW(tuned_FCN.parameters(), lr= 0.004038721483970501, weight_decay=0)

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=50,
    min_lr=0.0001
)

In [ ]:
metrics = {
    "epoch": [],
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "val_loss": [],
    "val_accuracy": []
}

In [ ]:

import os
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

os.makedirs("earlystop", exist_ok=True)

min_loss = float("inf")
stop_threshold = 0.93

for epoch in range(epochs):
    tuned_FCN.train()
    running_loss = 0.0
    total_samples = 0
    all_predictions = []
    all_labels = []

    # ===== Training =====
    for X, y in train_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        optimizer.zero_grad()

        outputs = tuned_FCN(X)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        predictions = outputs.argmax(dim=1)
        all_predictions.extend(predictions.detach().cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    epoch_loss = running_loss / total_samples

    accuracy = accuracy_score(all_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    # ===== Validation =====
    tuned_FCN.eval()

    val_running_loss = 0.0
    val_total_samples = 0
    val_predictions = []
    val_labels = []

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to("cuda")
            y = y.to("cuda")

            outputs = tuned_FCN(X)
            val_batch_loss = criterion(outputs, y)

            val_running_loss += val_batch_loss.item() * X.size(0)
            val_total_samples += X.size(0)

            predictions = outputs.argmax(dim=1)
            val_predictions.extend(predictions.cpu().numpy())
            val_labels.extend(y.cpu().numpy())

    val_loss = val_running_loss / val_total_samples
    val_accuracy = accuracy_score(val_labels, val_predictions)

    # ===== Save metrics =====
    metrics["epoch"].append(epoch)
    metrics["loss"].append(epoch_loss)
    metrics["accuracy"].append(accuracy)
    metrics["precision"].append(precision)
    metrics["recall"].append(recall)
    metrics["f1"].append(f1)
    metrics["val_loss"].append(val_loss)
    metrics["val_accuracy"].append(val_accuracy)

    # ===== Save best validation model =====
    if val_loss < min_loss:
        min_loss = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": tuned_FCN.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metrics": metrics,
            "val_loss": val_loss
        }, f"earlystop/fcn_tuned_best{val_loss}.pth")

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {epoch_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Train Acc: {accuracy:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

    # ===== Early stopping =====
    if val_accuracy >= stop_threshold:
        print(f"Stopping: validation accuracy reached {val_accuracy:.8f}_epoch_{epoch+1}")
        break

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv("FCN_tuned_metrics_early_stop.csv", index=False)


Epoch 1/2000 | Train Loss: 0.427387 | Val Loss: 4.284049 | Train Acc: 0.7926 | Val Acc: 0.4875 | Precision: 0.7939 | Recall: 0.7934 | F1: 0.7926
Epoch 2/2000 | Train Loss: 0.282247 | Val Loss: 1.605373 | Train Acc: 0.8821 | Val Acc: 0.5568 | Precision: 0.8825 | Recall: 0.8817 | F1: 0.8819
Epoch 3/2000 | Train Loss: 0.246094 | Val Loss: 1.364763 | Train Acc: 0.8994 | Val Acc: 0.5540 | Precision: 0.8998 | Recall: 0.8990 | F1: 0.8992
Epoch 4/2000 | Train Loss: 0.233778 | Val Loss: 0.918528 | Train Acc: 0.9022 | Val Acc: 0.6482 | Precision: 0.9027 | Recall: 0.9017 | F1: 0.9020
Epoch 5/2000 | Train Loss: 0.229594 | Val Loss: 0.423827 | Train Acc: 0.9022 | Val Acc: 0.7978 | Precision: 0.9025 | Recall: 0.9018 | F1: 0.9020
Epoch 6/2000 | Train Loss: 0.219058 | Val Loss: 0.257979 | Train Acc: 0.9096 | Val Acc: 0.8947 | Precision: 0.9098 | Recall: 0.9093 | F1: 0.9095
Epoch 7/2000 | Train Loss: 0.208093 | Val Loss: 0.203796 | Train Acc: 0.9145 | Val Acc: 0.9252 | Precision: 0.9146 | Recall: 0.914

In [66]:
len(test_loader)

21

In [67]:
tuned_FCN = FCN_tune(
    in_channels=1,
    num_classes=2,
    filters=[256, 128, 64],
    kernels=[8,10,6]
).to("cuda")
state_dict = torch.load(
    "earlystop/fcn_tuned_best0.19576737036995612.pth",
    map_location="cuda",
    weights_only=True
)
tuned_FCN.load_state_dict(state_dict["model_state_dict"])
tuned_FCN = tuned_FCN.to("cuda")
tuned_FCN.eval()

FCN_tune(
  (layers): Sequential(
    (0): Conv1d(1, 256, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.0, inplace=False)
    (4): Conv1d(256, 128, kernel_size=(10,), stride=(1,), padding=same)
    (5): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.0, inplace=False)
    (8): Conv1d(128, 64, kernel_size=(6,), stride=(1,), padding=same)
    (9): BatchNorm1d(64, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.0, inplace=False)
    (12): AdaptiveAvgPool1d(output_size=1)
    (13): Flatten(start_dim=1, end_dim=-1)
    (14): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [68]:
running_loss = 0.0
total_samples = 0
all_predictions = []
all_labels = []

test_metrics = {
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for X, y in test_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        # Forward pass
        outputs = tuned_FCN(X)
        loss = criterion(outputs, y)

        # Predictions
        predictions = torch.argmax(outputs, dim=1)

        # Accumulate loss
        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        # Collect predictions and labels
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
# Compute evaluation metrics
test_loss = running_loss / total_samples
accuracy = accuracy_score(all_labels, all_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

# Store metrics
test_metrics = {
    "loss": test_loss,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "predictions": all_predictions,
    "labels": all_labels
}

print(
    f"Test | "
    f"Loss: {test_loss:.4f} | "
    f"Accuracy: {accuracy:.4f} | "
    f"Precision: {precision:.4f} | "
    f"Recall: {recall:.4f} | "
    f"F1: {f1:.4f}"
)

Test | Loss: 0.2045 | Accuracy: 0.9227 | Precision: 0.9241 | Recall: 0.9238 | F1: 0.9227


#### try to run all datasets

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

root = Path("UCRArchive_2018")

binary_datasets = {}

for folder in root.iterdir():
    if not folder.is_dir():
        continue

    labels = set()

    for split in ["TRAIN", "TEST"]:
        file = folder / f"{folder.name}_{split}.tsv"

        if not file.exists():
            break

        # Only read the label column
        y = pd.read_csv(file, sep=r"\s+", header=None, usecols=[0])
        labels.update(y.iloc[:, 0].unique())
    else:
        if len(labels) == 2:
            binary_datasets[folder.name] = sorted(labels)

for name, labels in sorted(binary_datasets.items()):
    print(f"{name}: {labels}")

print(f"\nTotal binary datasets: {len(binary_datasets)}")

BeetleFly: [np.int64(1), np.int64(2)]
BirdChicken: [np.int64(1), np.int64(2)]
Chinatown: [np.int64(1), np.int64(2)]
Coffee: [np.int64(0), np.int64(1)]
Computers: [np.int64(1), np.int64(2)]
DistalPhalanxOutlineCorrect: [np.int64(0), np.int64(1)]
DodgerLoopGame: [np.int64(1), np.int64(2)]
DodgerLoopWeekend: [np.int64(1), np.int64(2)]
ECG200: [np.int64(-1), np.int64(1)]
ECGFiveDays: [np.int64(1), np.int64(2)]
Earthquakes: [np.int64(0), np.int64(1)]
FordA: [np.int64(-1), np.int64(1)]
FordB: [np.int64(-1), np.int64(1)]
FreezerRegularTrain: [np.int64(1), np.int64(2)]
FreezerSmallTrain: [np.int64(1), np.int64(2)]
GunPoint: [np.int64(1), np.int64(2)]
GunPointAgeSpan: [np.int64(1), np.int64(2)]
GunPointMaleVersusFemale: [np.int64(1), np.int64(2)]
GunPointOldVersusYoung: [np.int64(1), np.int64(2)]
Ham: [np.int64(1), np.int64(2)]
HandOutlines: [np.int64(0), np.int64(1)]
Herring: [np.int64(1), np.int64(2)]
HouseTwenty: [np.int64(1), np.int64(2)]
ItalyPowerDemand: [np.int64(1), np.int64(2)]
Lightni

In [2]:
from pathlib import Path

folder = Path("UCRArchive_2018")

folder_names = [p.name for p in folder.iterdir() if p.is_dir()]

In [3]:
folder_names

['ACSF1',
 'Adiac',
 'AllGestureWiimoteX',
 'AllGestureWiimoteY',
 'AllGestureWiimoteZ',
 'ArrowHead',
 'Beef',
 'BeetleFly',
 'BirdChicken',
 'BME',
 'Car',
 'CBF',
 'Chinatown',
 'ChlorineConcentration',
 'CinCECGTorso',
 'Coffee',
 'Computers',
 'CricketX',
 'CricketY',
 'CricketZ',
 'Crop',
 'DiatomSizeReduction',
 'DistalPhalanxOutlineAgeGroup',
 'DistalPhalanxOutlineCorrect',
 'DistalPhalanxTW',
 'DodgerLoopDay',
 'DodgerLoopGame',
 'DodgerLoopWeekend',
 'Earthquakes',
 'ECG200',
 'ECG5000',
 'ECGFiveDays',
 'ElectricDevices',
 'EOGHorizontalSignal',
 'EOGVerticalSignal',
 'EthanolLevel',
 'FaceAll',
 'FaceFour',
 'FacesUCR',
 'FiftyWords',
 'Fish',
 'FordA',
 'FordB',
 'FreezerRegularTrain',
 'FreezerSmallTrain',
 'Fungi',
 'GestureMidAirD1',
 'GestureMidAirD2',
 'GestureMidAirD3',
 'GesturePebbleZ1',
 'GesturePebbleZ2',
 'GunPoint',
 'GunPointAgeSpan',
 'GunPointMaleVersusFemale',
 'GunPointOldVersusYoung',
 'Ham',
 'HandOutlines',
 'Haptics',
 'Herring',
 'HouseTwenty',
 'Inli

In [4]:
from sklearn.preprocessing import LabelEncoder
# def readucr(filename):
#     data = np.loadtxt(filename, delimiter = ',')
#     Y = data[:,0]
#     X = data[:,1:]
#     return X, Y

root = Path("UCRArchive_2018")

def load_ucr_binary(folder, val_size=0.1, random_state=42):
    folder = Path(folder)
    name = folder.name

    train = pd.read_csv(
        folder / f"{name}_TRAIN.tsv",
        sep=r"\s+", header=None
    ).to_numpy()

    test = pd.read_csv(
        folder / f"{name}_TEST.tsv",
        sep=r"\s+", header=None
    ).to_numpy()

    X_train, y_train = train[:, 1:], train[:, 0]
    X_test, y_test = test[:, 1:], test[:, 0]

    # Encode labels consistently across splits
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y_train)

    if len(encoder.classes_) != 2:
        return None

    y_test = encoder.transform(y_test)

    # Split original training data into train/validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_train,
        y_train,
        test_size=val_size,
        random_state=random_state,
        stratify=y_train
    )

    # Convert into Conv1d input: (N, channels, length)
    X_train = X_train.astype(np.float32)[:, None, :]
    X_val = X_val.astype(np.float32)[:, None, :]
    X_test = X_test.astype(np.float32)[:, None, :]

    return X_train, X_val, X_test, y_train, y_val, y_test

def make_loader(X, y, batch_size=32, shuffle=False):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle
    )

for name in folder_names:
    folder = root / name
    data = load_ucr_binary(folder)

    if data is None:
        continue

    X_train, X_val, X_test, y_train, y_val, y_test = data
    train_loader = make_loader(X_train, y_train, shuffle=True)
    val_loader = make_loader(X_val, y_val)
    test_loader = make_loader(X_test, y_test)

    tuned_FCN = FCN_tune(
        in_channels=1,
        num_classes=2,
        filters=[256, 128, 64],
        kernels=[8,10,6]
    ).to("cuda")
    epochs = 2000
    optimizer = torch.optim.AdamW(tuned_FCN.parameters(), lr= 0.004038721483970501, weight_decay=0)

    criterion = nn.CrossEntropyLoss()

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=50,
        min_lr=0.0001
    )
    metrics = {
            "data": [],
            "epoch": [],
            "loss": [],
            "accuracy": [],
            "precision": [],
            "recall": [],
            "f1": [],
            "val_loss":[],
            "val_accuracy":[]
        }
    min_loss = float("inf")
    stop_threshold = 0.93

    for epoch in range(epochs):
        tuned_FCN.train()
        running_loss = 0.0
        total_samples = 0
        all_predictions = []
        all_labels = []

        # ===== Training =====
        for X, y in train_loader:
            X = X.to("cuda")
            y = y.to("cuda")

            optimizer.zero_grad()

            outputs = tuned_FCN(X)
            loss = criterion(outputs, y)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X.size(0)
            total_samples += X.size(0)

            predictions = outputs.argmax(dim=1)
            all_predictions.extend(predictions.detach().cpu().numpy())
            all_labels.extend(y.cpu().numpy())

        epoch_loss = running_loss / total_samples

        accuracy = accuracy_score(all_labels, all_predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )

        # ===== Validation =====
        tuned_FCN.eval()

        val_running_loss = 0.0
        val_total_samples = 0
        val_predictions = []
        val_labels = []

        with torch.no_grad():
            for X, y in val_loader:
                X = X.to("cuda")
                y = y.to("cuda")

                outputs = tuned_FCN(X)
                val_batch_loss = criterion(outputs, y)

                val_running_loss += val_batch_loss.item() * X.size(0)
                val_total_samples += X.size(0)

                predictions = outputs.argmax(dim=1)
                val_predictions.extend(predictions.cpu().numpy())
                val_labels.extend(y.cpu().numpy())

        val_loss = val_running_loss / val_total_samples
        val_accuracy = accuracy_score(val_labels, val_predictions)

        # ===== Save metrics =====
        metrics["data"].append(name)
        metrics["epoch"].append(epoch)
        metrics["loss"].append(epoch_loss)
        metrics["accuracy"].append(accuracy)
        metrics["precision"].append(precision)
        metrics["recall"].append(recall)
        metrics["f1"].append(f1)
        metrics["val_loss"].append(val_loss)
        metrics["val_accuracy"].append(val_accuracy)

        checkpoint_path = f"all_data/Fcn_tuned_best_{name}.pth"
        # ===== Save best validation model =====
        if val_loss < min_loss:
            min_loss = val_loss

            torch.save({
                "epoch": epoch,
                "model_state_dict": tuned_FCN.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "metrics": metrics,
                "val_loss": val_loss
            }, checkpoint_path)#f"all_data/Fcn_tuned_best{val_loss}_{folder}.pth")

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {epoch_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"Train Acc: {accuracy:.4f} | "
            f"Val Acc: {val_accuracy:.4f} | "
            f"Precision: {precision:.4f} | "
            f"Recall: {recall:.4f} | "
            f"F1: {f1:.4f}"
        )

        # ===== Early stopping =====
        if val_accuracy >= stop_threshold:
            print(f"Stopping: validation accuracy reached {val_accuracy:.8f}_epoch_{epoch+1}_{folder}")
            break
    # print(metrics)
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(f"all_data_metrics/FCN_tuned_metrics_early_stop_{name}.csv", index=False)
    # ===== Load best validation checkpoint =====
    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cuda",
            weights_only=True
        )

    except FileNotFoundError as e:
        print(f"Skipping dataset {name}: No checkpoint saved.")
        print(f"Error: {e}")
        continue

    tuned_FCN.load_state_dict(checkpoint["model_state_dict"])
    tuned_FCN.eval()

    # ===== Test evaluation =====
    test_running_loss = 0.0
    test_total_samples = 0
    test_predictions = []
    test_labels = []

    with torch.no_grad():
        for X, y in test_loader:
            X = X.to("cuda")
            y = y.to("cuda")

            outputs = tuned_FCN(X)
            loss = criterion(outputs, y)

            test_running_loss += loss.item() * X.size(0)
            test_total_samples += X.size(0)

            predictions = outputs.argmax(dim=1)

            test_predictions.extend(predictions.cpu().numpy())
            test_labels.extend(y.cpu().numpy())

    test_loss = test_running_loss / test_total_samples
    test_accuracy = accuracy_score(test_labels, test_predictions)

    test_precision, test_recall, test_f1, _ = (
        precision_recall_fscore_support(
            test_labels,
            test_predictions,
            labels=[0, 1],
            average="macro",
            zero_division=0
        )
    )

    print(
        f"\nDataset: {name}\n"
        f"Best Epoch: {checkpoint['epoch'] + 1}\n"
        f"Test Loss: {test_loss:.6f}\n"
        f"Test Accuracy: {test_accuracy:.4f}\n"
        f"Test Precision: {test_precision:.4f}\n"
        f"Test Recall: {test_recall:.4f}\n"
        f"Test F1: {test_f1:.4f}"
    )


c:\Users\96561\miniconda3\envs\UCR-pytorch\Lib\site-packages\torch\nn\modules\conv.py:380: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1105.)
  return F.conv1d(


Epoch 1/2000 | Train Loss: 0.693707 | Val Loss: 0.836854 | Train Acc: 0.5000 | Val Acc: 0.5000 | Precision: 0.2500 | Recall: 0.5000 | F1: 0.3333
Epoch 2/2000 | Train Loss: 0.675541 | Val Loss: 0.740804 | Train Acc: 0.4444 | Val Acc: 0.5000 | Precision: 0.4308 | Recall: 0.4444 | F1: 0.4156
Epoch 3/2000 | Train Loss: 0.611207 | Val Loss: 0.664448 | Train Acc: 0.8333 | Val Acc: 0.5000 | Precision: 0.8750 | Recall: 0.8333 | F1: 0.8286
Epoch 4/2000 | Train Loss: 0.552271 | Val Loss: 0.648726 | Train Acc: 0.8333 | Val Acc: 0.5000 | Precision: 0.8750 | Recall: 0.8333 | F1: 0.8286
Epoch 5/2000 | Train Loss: 0.507100 | Val Loss: 0.622926 | Train Acc: 0.8333 | Val Acc: 0.5000 | Precision: 0.8750 | Recall: 0.8333 | F1: 0.8286
Epoch 6/2000 | Train Loss: 0.476893 | Val Loss: 0.582866 | Train Acc: 0.8333 | Val Acc: 1.0000 | Precision: 0.8750 | Recall: 0.8333 | F1: 0.8286
Stopping: validation accuracy reached 1.00000000_epoch_6_UCRArchive_2018\BeetleFly

Dataset: BeetleFly
Best Epoch: 6
Test Loss: 0.